<h1>RAZ Systems </h1>

Building a multi-agent AI system that collaborates to create and send effective **customer win-back** emails for **Loopline Analytics**, a SaaS analytics company trying to re-engage churned customers.

Details of the requirements:

| Problem | Solution in Your System |
| --- | --- |
| One AI response may be weak or generic | Multiple win-back agents generate different styles of emails |
| Hard to choose the best email | Retention Manager compares and selects the strongest draft |
| Unsafe or sensitive input | Guardrail Agent validates requests before processing |
| Email formatting is manual | HTML Converter Agent automates formatting |
| Subject lines impact open rates | Subject Writer Agent optimizes subject generation |
| Sending emails requires code integration | Send Email Function automates delivery |


| Concept | Meaning |
| --- | --- |
| Agent | AI worker |
| Tool | External function/API |
| Handoff | Transfer between agents |
| Runner | Executes workflow |
| Trace | Monitoring/debugging |
| Guardrail | Safety/validation |
| Structured Output | Reliable schema output |
| Multi-Agent System | Multiple agents collaborating |

Now we get to more detail:

1. Different models
2. Structured Outputs
3. Guardrails

**Your task:** fill in every `# TODO`. A full solution is at the end of this notebook -- try not to peek until you've had a go.

### This time, the guardrail is different

In the lesson, the guardrail checked whether the message contained a **personal name**. Here, your guardrail will check whether the request mentions a **named competitor** -- Loopline's legal team doesn't want automated emails that reference competitors by name. Same pattern, different check.

| Agent / Component | Type | Specialization / Responsibility |
| --- | --- | --- |
| Guardrail Agent | Safety / Validation Agent | Detects competitor names, blocks risky requests |
| Retention Manager | Reasoning / Orchestrator Agent | Calls 3 win-back agents, compares drafts, chooses best email |
| Win-back Agent 1 | Specialist Agent | Empathetic, relationship-focused win-back email |
| Win-back Agent 2 | Specialist Agent | Short, incentive-led win-back email |
| Win-back Agent 3 | Specialist Agent | Data-driven, product-update-focused win-back email |
| Email Manager Agent | Workflow / Orchestrator Agent | Manages subject generation, HTML conversion, and sending |
| Subject Writer Agent | Specialist Agent | Generates re-engaging win-back subject lines |
| HTML Converter Agent | Formatter / Specialist Agent | Converts plain-text email into HTML email format |
| Send Email Function | Function Tool | Sends final email using SendGrid |


In [ ]:
# --- Imports ---
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from pydantic import BaseModel

In [ ]:
load_dotenv(override=True)

In [ ]:
# --- Check which provider keys you have set ---

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

## Part 1 -- Three win-back personas, on three different models

**TODO:** write three win-back email personas for Loopline Analytics:
- Agent 1: empathetic, relationship-focused
- Agent 2: short, incentive-led (e.g. a discount or extended trial)
- Agent 3: data-driven, focused on new product updates they've missed

It's easy to use any model with an OpenAI-compatible endpoint -- we'll run these three on **DeepSeek**, **Gemini**, and **Llama 3.3** respectively, the same way the lesson did.

In [ ]:
# --- TODO: three win-back personas ---

instructions1 = ___  # TODO: empathetic, relationship-focused win-back persona for Loopline Analytics

instructions2 = ___  # TODO: short, incentive-led win-back persona for Loopline Analytics

instructions3 = ___  # TODO: data-driven, product-update-focused win-back persona for Loopline Analytics

In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

In [ ]:
deepseek_client = AsyncOpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

deepseek_model = OpenAIChatCompletionsModel(model="deepseek-chat", openai_client=deepseek_client)
gemini_model = OpenAIChatCompletionsModel(model="gemini-2.0-flash", openai_client=gemini_client)
llama3_3_model = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client=groq_client)

In [ ]:
# --- TODO: create the three win-back agents on three different models ---

winback_agent1 = Agent(name=___, instructions=instructions1, model=___)  # TODO: name + deepseek_model
winback_agent2 = Agent(name=___, instructions=instructions2, model=___)  # TODO: name + gemini_model
winback_agent3 = Agent(name=___, instructions=instructions3, model=___)  # TODO: name + llama3_3_model

## Part 2 -- Agent-as-tool + function tools

**Reminder -- Function Tool vs Agent Tool:**

| Feature | Function Tool | Agent Tool |
| --- | --- | --- |
| You write Python logic | Yes | No |
| Uses LLM internally | Optional | Always |
| Deterministic | Usually | No |
| Good for APIs/calculations | Yes | Sometimes |
| Good for reasoning/writing | No | Yes |
| Requires `@function_tool` | Yes | No |
| Uses `.as_tool()` | No | Yes |

**TODO:** convert each win-back agent into a tool.

In [ ]:
# --- TODO: convert each win-back agent into a tool ---

description = ___  # TODO: e.g. "Write a customer win-back email"

tool1 = winback_agent1.as_tool(tool_name=___, tool_description=description)  # TODO
tool2 = winback_agent2.as_tool(tool_name=___, tool_description=description)  # TODO
tool3 = winback_agent3.as_tool(tool_name=___, tool_description=description)  # TODO

In [ ]:
# --- TODO: function tool that actually sends the HTML email ---

@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ ___ """  # TODO: docstring -- sends a win-back email with subject + HTML body
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email(___)  # TODO: your verified sender
    to_email = To(___)       # TODO: your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

**TODO:** build the Subject Writer and HTML Converter sub-agents, and the Email Manager that uses both plus `send_html_email`.

In [ ]:
# --- TODO: Subject Writer + HTML Converter ---

subject_instructions = ___  # TODO: writes catchy, re-engaging subject lines for a win-back email
html_instructions = ___     # TODO: converts plain-text win-back email body to simple, clean HTML

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a win-back email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a text email body to an HTML email body")

In [ ]:
email_tools = [subject_tool, html_tool, send_html_email]

# --- TODO: Email Manager instructions ---
instructions = ___  # TODO: receives an email body, uses subject_writer, then html_converter,
# then send_html_email, in that order

emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=email_tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it",
)

## Part 3 -- Retention Manager (orchestrator with tools + handoff)

**TODO:** write the Retention Manager's instructions: generate all three win-back drafts, evaluate, then hand off exactly ONE winning draft to the Email Manager.

In [ ]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]

retention_manager_instructions = """
___
"""  # TODO: You are a Retention Manager at Loopline Analytics. Goal: find the single best
# win-back email using the win-back agent tools, then hand off ONLY the winning draft to
# the 'Email Manager' agent. Never write the email yourself, never hand off more than one.

retention_manager = Agent(
    name="Retention Manager",
    instructions=retention_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini")

message = ___  # TODO: e.g. "Send a win-back email to a churned customer named Dear Customer, from Loopline Support"

with trace("Loopline win-back"):
    result = await Runner.run(retention_manager, message)

## Check out the trace:

https://platform.openai.com/traces

## Part 4 -- Guardrail: block requests that mention a competitor by name

**New concept -- Structured Output:** instead of free text, the guardrail agent returns data in a fixed shape (a Pydantic model), so your code can reliably check a boolean field rather than parsing prose.

**TODO:**
1. Define a `CompetitorMentionOutput` Pydantic model with `mentions_competitor: bool` and `competitor_name: str`
2. Create a guardrail agent that checks for competitor names
3. Wrap it with `@input_guardrail`
4. Attach it to a new `careful_retention_manager` via `input_guardrails=[...]`

In [ ]:
# --- TODO: structured output model + guardrail agent ---

class CompetitorMentionOutput(BaseModel):
    mentions_competitor: ___   # TODO: bool
    competitor_name: ___        # TODO: str

guardrail_agent = Agent(
    name="Competitor check",
    instructions=___,  # TODO: "Check if the user is naming a specific competitor analytics product in what they want you to do."
    output_type=CompetitorMentionOutput,
    model="gpt-4o-mini",
)

In [ ]:
# --- TODO: wrap the guardrail agent as an @input_guardrail ---

@input_guardrail
async def guardrail_against_competitor(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    mentions_competitor = result.final_output.___   # TODO: which field?
    return GuardrailFunctionOutput(
        output_info={"found_competitor": result.final_output},
        tripwire_triggered=___,   # TODO
    )

Main entry point -- this run should get BLOCKED:

In [ ]:
# --- TODO: attach the guardrail and try a message that mentions a competitor ---

careful_retention_manager = Agent(
    name="Retention Manager",
    instructions=retention_manager_instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model="gpt-4o-mini",
    input_guardrails=[___],  # TODO: guardrail_against_competitor
)

message = ___  # TODO: a message that mentions a named competitor, e.g.
# "Send a win-back email pointing out we are better than RivalMetrics"

with trace("Protected Loopline win-back"):
    result = await Runner.run(careful_retention_manager, message)

This run should succeed -- no competitor mentioned:

In [ ]:
message = ___  # TODO: a clean message with no competitor name

with trace("Protected Loopline win-back"):
    result = await Runner.run(careful_retention_manager, message)

---
# Solution

No peeking until you've tried it yourself!

In [ ]:
# === Setup ===
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from pydantic import BaseModel

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

In [ ]:
# === Part 1 -- personas + multi-model agents ===

instructions1 = (
    "You are an empathetic customer success agent at Loopline Analytics, a SaaS analytics company. "
    "You write warm, relationship-focused win-back emails to churned customers, acknowledging their "
    "reasons for leaving and inviting an open conversation."
)

instructions2 = (
    "You are a growth-focused agent at Loopline Analytics, a SaaS analytics company. "
    "You write short, incentive-led win-back emails offering a discount or extended trial to return."
)

instructions3 = (
    "You are a product-focused agent at Loopline Analytics, a SaaS analytics company. "
    "You write data-driven win-back emails highlighting new product features the customer has missed."
)

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

deepseek_client = AsyncOpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

deepseek_model = OpenAIChatCompletionsModel(model="deepseek-chat", openai_client=deepseek_client)
gemini_model = OpenAIChatCompletionsModel(model="gemini-2.0-flash", openai_client=gemini_client)
llama3_3_model = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client=groq_client)

winback_agent1 = Agent(name="DeepSeek Win-back Agent", instructions=instructions1, model=deepseek_model)
winback_agent2 = Agent(name="Gemini Win-back Agent", instructions=instructions2, model=gemini_model)
winback_agent3 = Agent(name="Llama3.3 Win-back Agent", instructions=instructions3, model=llama3_3_model)

In [ ]:
# === Part 2 -- tools ===

description = "Write a customer win-back email"

tool1 = winback_agent1.as_tool(tool_name="winback_agent1", tool_description=description)
tool2 = winback_agent2.as_tool(tool_name="winback_agent2", tool_description=description)
tool3 = winback_agent3.as_tool(tool_name="winback_agent3", tool_description=description)

@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out a win-back email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("pashaajaz@gmail.com")  # Change to your verified sender
    to_email = To("pashaajaz@gmail.com")        # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

subject_instructions = (
    "You can write a subject for a customer win-back email. "
    "You are given a message and need to write a subject likely to get a response."
)

html_instructions = (
    "You can convert a text email body to an HTML email body. "
    "You are given a text email body which might have some markdown, and you need to convert it "
    "to an HTML email body with a simple, clear, compelling layout."
)

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a win-back email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a text email body to an HTML email body")

email_tools = [subject_tool, html_tool, send_html_email]

instructions = (
    "You are an email formatter and sender. You receive the body of an email to be sent. "
    "You first use the subject_writer tool to write a subject for the email, then use the "
    "html_converter tool to convert the body to HTML. Finally, you use the send_html_email tool "
    "to send the email with the subject and HTML body."
)

emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=email_tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it",
)

In [ ]:
# === Part 3 -- Retention Manager ===

tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]

retention_manager_instructions = """
You are a Retention Manager at Loopline Analytics. Your goal is to find the single best win-back
email using the win-back agent tools.

Follow these steps carefully:
1. Generate Drafts: Use all three win-back agent tools to generate three different email drafts.
Do not proceed until all three drafts are ready.

2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of
which one is most likely to win back the customer. You can use the tools multiple times if you're
not satisfied with the results from the first try.

3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email
Manager will take care of formatting and sending.

Crucial Rules:
- You must use the win-back agent tools to generate the drafts -- do not write them yourself.
- You must hand off exactly ONE email to the Email Manager -- never more than one.
"""

retention_manager = Agent(
    name="Retention Manager",
    instructions=retention_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini")

message = "Send a win-back email to a churned customer, addressed Dear Customer, from Loopline Support"

with trace("Loopline win-back"):
    result = await Runner.run(retention_manager, message)

In [ ]:
# === Part 4 -- structured output + guardrail ===

class CompetitorMentionOutput(BaseModel):
    mentions_competitor: bool
    competitor_name: str

guardrail_agent = Agent(
    name="Competitor check",
    instructions="Check if the user is naming a specific competitor analytics product in what they want you to do.",
    output_type=CompetitorMentionOutput,
    model="gpt-4o-mini",
)

@input_guardrail
async def guardrail_against_competitor(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    mentions_competitor = result.final_output.mentions_competitor
    return GuardrailFunctionOutput(
        output_info={"found_competitor": result.final_output},
        tripwire_triggered=mentions_competitor,
    )

In [ ]:
# === This run gets BLOCKED -- the message names a competitor ===

careful_retention_manager = Agent(
    name="Retention Manager",
    instructions=retention_manager_instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model="gpt-4o-mini",
    input_guardrails=[guardrail_against_competitor],
)

message = "Send a win-back email pointing out we are better than RivalMetrics"

with trace("Protected Loopline win-back"):
    result = await Runner.run(careful_retention_manager, message)

In [ ]:
# === This run succeeds -- no competitor named ===

message = "Send a win-back email to a churned customer, addressed Dear Customer, from Loopline Support"

with trace("Protected Loopline win-back"):
    result = await Runner.run(careful_retention_manager, message)